# Run Real Video VAE Latent Probe

## 00 Runtime Identity And User Config (`00_runtime_identity_and_user_config`)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from paper_workflow.notebook_utils import real_video_vae_latent_probe_workflow as probe_workflow

RUN_MODE = 'formal'
RUNTIME_PROFILE = 'formal'
WORKFLOW_KEY = 'real_video_vae_latent_probe_completion_formal'
STEP_KEY = 'step02_run_real_video_vae_latent_probe'
FAMILY_ID = 'real_video_vae_latent_probe__formal__davis2017_trainval480p__utc_time__short_commit'
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001'
REPO_ROOT = Path.cwd().resolve()
DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_RUNTIME_ROOT = Path('/content/TSTW_runtime')
PROCESSED_DATASET_ROOT = DRIVE_ROOT / 'TSTW' / 'datasets' / 'processed' / PROCESSED_DATASET_KEY
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'datasets' / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = LOCAL_DATASET_ROOT / 'dataset_manifest.json'
LOCAL_MODEL_ROOT = LOCAL_RUNTIME_ROOT / 'session_models' / 'autoencoder_kl'
FAMILY_ROOT = DRIVE_ROOT / 'TSTW' / 'results' / 'families' / FAMILY_ID
RUN_ROOT = LOCAL_RUNTIME_ROOT / 'runs' / 'real_video_vae_latent_probe_formal'
RUNTIME_CONFIG_PATH = RUN_ROOT / 'artifacts' / 'runtime_config.json'
SESSION_MODEL_MANIFEST_PATH = RUN_ROOT / 'artifacts' / 'session_model_manifest.json'
MODEL_ID = 'stabilityai/sd-vae-ft-mse'
REQUIRE_FORMAL_PASS = True

## 01 Mount Google Drive (`01_mount_google_drive`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 02 Read Processed Dataset Registry (`02_read_processed_dataset_registry`)

In [ ]:
if not PROCESSED_DATASET_ROOT.exists():
    raise FileNotFoundError(PROCESSED_DATASET_ROOT)
print({
    'processed_dataset_root': str(PROCESSED_DATASET_ROOT),
    'processed_dataset_key': PROCESSED_DATASET_KEY,
})

## 03 Prepare Local Runtime Workspace (`03_prepare_local_runtime_workspace`)

In [ ]:
runtime_workspace_handoff = probe_workflow.prepare_probe_runtime_workspace(
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    local_dataset_root=LOCAL_DATASET_ROOT,
    family_root=FAMILY_ROOT,
    run_root=RUN_ROOT,
    copy_processed_dataset=True,
)
PROCESSED_DATASET_MANIFEST = Path(runtime_workspace_handoff['local_dataset_manifest_path'])
if not PROCESSED_DATASET_MANIFEST.exists():
    raise FileNotFoundError(PROCESSED_DATASET_MANIFEST)
print(runtime_workspace_handoff)

## 04 Clone Or Update Repository (`04_clone_or_update_repository`)

In [ ]:
repo_env = dict(os.environ)
repo_env['PYTHONPATH'] = str(REPO_ROOT)
GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'],
    text=True,
    env=repo_env,
    cwd=REPO_ROOT,
).strip()
subprocess.run(['git', 'status', '--short'], check=True, env=repo_env, cwd=REPO_ROOT)

## 05 Install Runtime Dependencies (`05_install_runtime_dependencies`)

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-e',
        '.',
        'pytest',
        'diffusers',
        'accelerate',
        'transformers',
        'safetensors',
        'huggingface_hub',
        'lpips',
        'pytorch-msssim',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 06 Prepare Session Model (`06_prepare_session_model`)

In [ ]:
session_model_manifest = probe_workflow.prepare_probe_session_model(
    model_id=MODEL_ID,
    local_model_root=LOCAL_MODEL_ROOT,
    session_manifest_path=SESSION_MODEL_MANIFEST_PATH,
    revision='main',
)
LOCAL_MODEL_ROOT = Path(session_model_manifest['models'][0]['local_path'])
if not LOCAL_MODEL_ROOT.exists():
    raise FileNotFoundError(LOCAL_MODEL_ROOT)
assert session_model_manifest['model_policy'] == 'session_only_no_drive_model_storage'

## 07 Write Runtime Config (`07_write_runtime_config`)

In [ ]:
runtime_config_handoff = probe_workflow.write_probe_runtime_config(
    runtime_config_path=RUNTIME_CONFIG_PATH,
    execution_environment='colab',
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_dataset_root=LOCAL_DATASET_ROOT,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    vae_model_local_path=LOCAL_MODEL_ROOT,
    dataset_manifest_path=PROCESSED_DATASET_MANIFEST,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    extra_config={
        'family_id': FAMILY_ID,
        'workflow_key': WORKFLOW_KEY,
        'step_key': STEP_KEY,
        'git_commit': GIT_COMMIT,
        'runtime_manifest_overrides': {
            'family_root': str(FAMILY_ROOT),
            'session_model_manifest_path': str(SESSION_MODEL_MANIFEST_PATH),
        },
    },
)
print(runtime_config_handoff)

## 08 Check GPU And Runtime (`08_check_gpu_and_runtime`)

In [ ]:
runtime_check_report = probe_workflow.run_probe_runtime_preflight(
    run_mode=RUN_MODE,
    local_dataset_root=LOCAL_DATASET_ROOT,
    local_model_dirs=[LOCAL_MODEL_ROOT],
)
print(runtime_check_report)

## 09 Verify Repository Contract (`09_verify_repository_contract`)

In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], check=True, env=repo_env, cwd=REPO_ROOT)
subprocess.run([sys.executable, 'tools/harness/run_all_audits.py'], check=True, env=repo_env, cwd=REPO_ROOT)

## 10 Run Smoke Tests (`10_run_smoke_tests`)

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        '-m',
        'smoke',
        'tests/integration/test_real_video_vae_encode_decode_smoke.py',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 11 Run Real Video VAE Latent Formal (`11_run_real_video_vae_latent_formal`)

In [ ]:
probe_workflow.run_probe_runner(
    run_root=RUN_ROOT,
    run_mode=RUN_MODE,
    runtime_profile=RUNTIME_PROFILE,
    runtime_config_path=RUNTIME_CONFIG_PATH,
    dataset_manifest=PROCESSED_DATASET_MANIFEST,
    python_executable=sys.executable,
)

## 12 Rebuild Tables And Reports (`12_rebuild_tables_and_reports`)

In [ ]:
probe_workflow.rebuild_probe_tables_and_reports(run_root=RUN_ROOT)

## 13 Check Real Video VAE Latent Outputs (`13_check_real_video_vae_latent_outputs`)

In [ ]:
formal_validation_summary = probe_workflow.check_probe_outputs(
    run_root=RUN_ROOT,
    construction_phase='real_video_vae_latent_probe',
    run_mode=RUN_MODE,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
)
if not formal_validation_summary['status']:
    raise RuntimeError(formal_validation_summary)

## 14 Package Family Results (`14_package_family_results`)

In [ ]:
package_payload = probe_workflow.package_probe_family_results(
    run_root=RUN_ROOT,
    family_root=FAMILY_ROOT,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    formal_validation_summary=formal_validation_summary,
    drive_root=DRIVE_ROOT,
    family_id=FAMILY_ID,
    workflow_key=WORKFLOW_KEY,
    step_key=STEP_KEY,
    run_mode=RUN_MODE,
)
drive_archive_path = package_payload['drive_archive_path']
compat_pack_root = package_payload['compat_pack_root']
print({
    'drive_archive_path': drive_archive_path,
    'compat_pack_root': compat_pack_root,
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
})

## 15 Print Final Family Summary (`15_print_final_family_summary`)

In [ ]:
print({
    'family_id': FAMILY_ID,
    'run_root': str(RUN_ROOT),
    'family_root': str(FAMILY_ROOT),
    'formal_validation_summary': formal_validation_summary,
    'drive_archive_path': str(drive_archive_path),
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
})